# Figures 3, 4, and S4 — Temporal Trend Clustering & Comparison

This notebook generates three figure sets from canonical daily frequencies.

- **Figure 3:** DTW + KMeans language clusters for 2014 (k=5) and 2022 (k=6), plotted as cluster trend panels.
- **Figure 4:** 2014 vs 2022 raw-frequency overlays (log scale), with 2014 shifted by **+8 years, −3 days** for event-aligned comparison.
- **Figure S4:** Cluster diagnostics (WCSS elbow + Davies–Bouldin) for k=1..15.

## Input
- `config.CHOSEN_WORDS_DAILY_FILE` (`data/processed/chosen_words_daily.csv`)

## Outputs
- `outputs/figures/Fig.3_kmeans_clustering_2014/`
- `outputs/figures/Fig.3_kmeans_clustering_2022/`
- `outputs/figures/Fig.4_trends_2014_vs_2022/`
- `outputs/figures/Fig.S4_cluster_evaluations/`

(saved as `pdf`, `png`, `svg`)

**Prerequisite:** run `02_combine_data.ipynb` first.

In [ ]:
# Import required libraries for data processing, visualization, clustering, and analysis
import sys
sys.path.insert(0, '..')

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score
from dtaidistance import dtw

# Import configuration: paths, language lists, and dataset locations
from config import WORD_FORMS_ALL, CHOSEN_WORDS_DAILY_FILE, FIGURES_DIR, LANGUAGE_ORDER

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_white'

In [ ]:
# Define temporal reference points and output directories for all figures

# Key dates for analysis: clustering and comparison windows are centered on these dates
CENTRAL_2014_FIG4 = pd.to_datetime('2014-02-27')      # Center date for 2014 Fig.4 comparison
CENTRAL_2014_CLUSTER = pd.to_datetime('2014-02-18')   # Center date for 2014 Fig.3 clustering
CENTRAL_2022 = pd.to_datetime('2022-02-24')           # Center date for 2022 (Ukraine invasion)

# Date shift applied to 2014 data when overlaying with 2022 for comparison
FIG4_2014_SHIFT = pd.DateOffset(years=8, days=-3)     # Shift 2014 by +8 years, -3 days

# Create output directories for each figure
FIG3_OUT_2014 = FIGURES_DIR / 'Fig.3_kmeans_clustering_2014'
FIG3_OUT_2022 = FIGURES_DIR / 'Fig.3_kmeans_clustering_2022'
FIG4_OUT = FIGURES_DIR / 'Fig.4_trends_2014_vs_2022'
FIGS4_OUT = FIGURES_DIR / 'Fig.S4_cluster_evaluations'

for d in [FIG3_OUT_2014, FIG3_OUT_2022, FIG4_OUT, FIGS4_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# Language order for Fig.4 subplots (determines subplot layout)
FIG4_LANGUAGE_ORDER = [
    'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean', 'Persian', 'Turkish',
    'Indonesian', 'Urdu', 'Vietnamese', 'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian',
    'Polish', 'Swedish', 'Czech', 'Spanish', 'Danish', 'English', 'Dutch', 'Norwegian',
    'Finnish', 'French', 'Italian', 'German'
]

print('Output folders ready.')

In [ ]:
# Load daily time-series data and create language-level aggregates

# Read canonical word frequency data (pre-filtered to Canonical==1 words)
df_all = pd.read_csv(CHOSEN_WORDS_DAILY_FILE)
df_all['date'] = pd.to_datetime(df_all['date'])

# Build ISO code → Language name mapping from the input query metadata
iso_to_name = (
    pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language'])
    .drop_duplicates()
    .set_index('ISO')['Language']
)

# Map language ISO codes to human-readable language names
df_all['language'] = df_all['language_ISO'].map(iso_to_name)
df_all = df_all.dropna(subset=['language']).copy()

# Aggregate all query variants to daily language-level totals
# (sum across all queries within each language per day)
df_lang_day = (
    df_all.groupby(['date', 'language'], as_index=False)[['freq', 'count', 'count_no_rt', 'freq_no_rt']]
    .sum()
)

print(df_lang_day[['date', 'language', 'freq']].head(3))
print(f'Languages: {df_lang_day.language.nunique()}')

In [ ]:
# Define time-window extraction and normalization functions

def slice_window(df, center_date, weeks_before, weeks_after):
    """Extract a time window around a center date.
    
    Args:
        df: DataFrame with 'date' column
        center_date: pandas Timestamp marking window center
        weeks_before: weeks to include before center_date
        weeks_after: weeks to include after center_date
    
    Returns:
        Filtered DataFrame within [center_date - weeks_before, center_date + weeks_after]
    """
    start = center_date - pd.Timedelta(weeks=weeks_before)
    end = center_date + pd.Timedelta(weeks=weeks_after)
    return df[(df['date'] >= start) & (df['date'] <= end)].copy()

def add_local_normalized_freq(df, freq_col='freq', lang_col='language'):
    """Normalize frequencies by local (per-language) mean.
    
    Computes normalized_loc_freq = freq / (mean freq within window for that language).
    Also creates log-transformed versions for later plotting.
    
    Args:
        df: DataFrame with 'date' and language-identifying column
        freq_col: column name containing raw frequencies
        lang_col: column name identifying languages
    
    Returns:
        DataFrame with added normalized_loc_freq, normalized_loc_freq_log, freq_log columns
    """
    out = df.copy()
    local_mean = out.groupby(lang_col)[freq_col].transform('mean')
    out['normalized_loc_freq'] = out[freq_col] / local_mean
    out['normalized_loc_freq_log'] = np.log10(out['normalized_loc_freq'])
    out['freq_log'] = np.log10(out[freq_col].replace(0, np.nan))
    return out

# Prepare data for Fig.3 clustering: use different windows for 2014 and 2022
# (2014 clustering learned on -1w/+3w window, 2022 on ±4w window, per original study design)
DF2014_cluster = add_local_normalized_freq(slice_window(df_lang_day, CENTRAL_2014_CLUSTER, 1, 3))
DF2022_cluster = add_local_normalized_freq(slice_window(df_lang_day, CENTRAL_2022, 4, 4))

# Prepare data for Fig.4 comparison: both years use ±4w window
# (2014 data is then shifted +8 years, -3 days to overlay with 2022)
DF2014_fig4 = slice_window(df_lang_day, CENTRAL_2014_FIG4, 4, 4)
DF2022_fig4 = slice_window(df_lang_day, CENTRAL_2022, 4, 4)
DF2014_fig4['date'] = DF2014_fig4['date'] + FIG4_2014_SHIFT

print('Fig3 windows:', DF2014_cluster['date'].min().date(), DF2014_cluster['date'].max().date(), '|', DF2022_cluster['date'].min().date(), DF2022_cluster['date'].max().date())
print('Fig4 windows:', DF2014_fig4['date'].min().date(), DF2014_fig4['date'].max().date(), '|', DF2022_fig4['date'].min().date(), DF2022_fig4['date'].max().date())

In [ ]:
# Define helper functions for DTW-based clustering and evaluation

def make_pivot(df, value_col='normalized_loc_freq'):
    """Pivot time-series data into language × date matrix.
    
    Each row is a language, each column is a date, cells contain the normalized frequency.
    Missing values are filled with 0.
    
    Args:
        df: DataFrame with 'language', 'date', and value_col columns
        value_col: column to pivot (default: normalized_loc_freq)
    
    Returns:
        Pivot table (DataFrame) with languages as rows, dates as columns
    """
    pivot = df.pivot(index='language', columns='date', values=value_col).fillna(0.0)
    return pivot

def dtw_kmeans_labels(df_pivot, n_clusters, random_state=0):
    """Compute DTW distance matrix and perform KMeans clustering.
    
    Computes pairwise Dynamic Time Warping distances between all language time-series,
    then fits KMeans on this distance matrix to identify n_clusters groups.
    
    Args:
        df_pivot: Pivot table (languages × dates) with normalized frequencies
        n_clusters: Number of clusters to extract
        random_state: Random seed for reproducibility
    
    Returns:
        Tuple: (cluster_labels, dtw_distance_matrix)
    """
    data_array = df_pivot.values
    dtw_distance_matrix = dtw.distance_matrix(data_array)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(dtw_distance_matrix)
    return labels, dtw_distance_matrix

def evaluate_cluster_curve(df_pivot, max_clusters=15):
    """Evaluate cluster quality across a range of k values.
    
    For each k from 1 to max_clusters, computes:
    - WCSS (within-cluster sum of squares)
    - Davies-Bouldin index (lower is better)
    
    Used to determine optimal cluster count via elbow method or DB minimization.
    
    Args:
        df_pivot: Pivot table (languages × dates) with normalized frequencies
        max_clusters: Maximum k to evaluate (default: 15)
    
    Returns:
        Tuple: (wcss_list, davies_bouldin_scores)
        - wcss_list: WCSS values for k=1..max_clusters
        - davies_bouldin_scores: DB scores for k=1..max_clusters (k=1 is None)
    """
    data_array = df_pivot.values
    dtw_distance_matrix = dtw.distance_matrix(data_array)

    wcss = []
    db_scores = [None]

    for k in range(1, max_clusters + 1):
        model = KMeans(n_clusters=k, random_state=0, n_init=10)
        if k == 1:
            model.fit(df_pivot)
            wcss.append(model.inertia_)
        else:
            labels = model.fit_predict(dtw_distance_matrix)
            wcss.append(model.inertia_)
            db_scores.append(davies_bouldin_score(df_pivot, labels))

    return wcss, db_scores

In [ ]:
# Generate Figure S4: Cluster evaluation curves (WCSS and Davies-Bouldin index)

# Create pivot tables from clustering windows
pivot_2014 = make_pivot(DF2014_cluster)
pivot_2022 = make_pivot(DF2022_cluster)

# Evaluate cluster quality for k=1..15 clusters
wcss_2014, db_2014 = evaluate_cluster_curve(pivot_2014, max_clusters=15)
wcss_2022, db_2022 = evaluate_cluster_curve(pivot_2022, max_clusters=15)

def plot_s4(wcss, db_scores, title, save_base):
    """Plot cluster evaluation metrics (WCSS and Davies-Bouldin index).
    
    Creates side-by-side plots showing:
    - Left: WCSS elbow curve
    - Right: Davies-Bouldin index (lower is better)
    
    Args:
        wcss: List of WCSS values (k=1..max_clusters)
        db_scores: List of DB scores (k=1..max_clusters, with k=1 as None)
        title: Figure title and output filename
        save_base: Base directory for saving (will create pdf/png/svg subdirs)
    
    Returns:
        Matplotlib figure object
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].plot(range(1, len(wcss) + 1), wcss, marker='o')
    axes[0].set_title('WCSS', fontsize=16)
    axes[0].set_xlabel('Number of Clusters', fontsize=16)
    axes[0].set_xticks([2, 5, 8, 11, 14])
    axes[0].grid(True)

    axes[1].plot(range(2, len(db_scores) + 1), db_scores[1:], marker='o')
    axes[1].set_title('Davies-Bouldin Index', fontsize=16)
    axes[1].set_xlabel('Number of Clusters', fontsize=16)
    axes[1].set_xticks([2, 5, 8, 11, 14])
    axes[1].grid(True)

    fig.suptitle(title, fontsize=16)
    fig.tight_layout()

    # Save in multiple formats
    for fmt in ['pdf', 'png', 'svg']:
        out_dir = save_base / fmt
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / f'{title}.{fmt}', format=fmt, dpi=400, bbox_inches='tight')

    return fig

# Generate and save S4 plots for both years
_ = plot_s4(wcss_2014, db_2014, 'Clustertests_2014', FIGS4_OUT / '2014')
_ = plot_s4(wcss_2022, db_2022, 'Clustertests_2022', FIGS4_OUT / '2022')
plt.show()

In [ ]:
# Generate Figure 3: Trend clusters visualized as 3×3 subplots

# Compute DTW-based cluster labels
# (2014: 5 clusters learned on -1w/+3w window; 2022: 6 clusters learned on ±4w window)
labels_2014, _ = dtw_kmeans_labels(pivot_2014, n_clusters=5)
labels_2022_raw, _ = dtw_kmeans_labels(pivot_2022, n_clusters=6)

# Reload data for plotting (2014 on ±4w window for display, matching figure caption)
# while keeping clustering labels from original -1w/+3w window
DF2014_plot = add_local_normalized_freq(slice_window(df_lang_day, CENTRAL_2014_CLUSTER, 4, 4))
DF2022_plot = DF2022_cluster.copy()  # already ±4w

pivot_2014_plot = make_pivot(DF2014_plot)
pivot_2022_plot = make_pivot(DF2022_plot)

def remap_clusters_by_representatives(labels, language_index, target_representatives):
    """Stabilize cluster IDs by matching language composition to representative sets.
    
    KMeans assigns arbitrary cluster IDs that can vary across runs. This function
    remaps raw cluster IDs to semantically stable target IDs by comparing each
    raw cluster's language membership to predefined representative language sets
    from the original paper.
    
    Example: If raw cluster 2 contains {Indonesian, Korean, Portuguese, Urdu},
    it gets remapped to target ID 1 (which expects those languages).
    
    Args:
        labels: Array of raw cluster IDs (0..n_clusters-1)
        language_index: Array of language names corresponding to each row
        target_representatives: Dict mapping target IDs to sets of representative languages
    
    Returns:
        Tuple: (remapped_labels, raw_to_target_mapping_dict)
    """
    labels = np.asarray(labels)
    languages = np.asarray(language_index)
    raw_clusters = sorted(np.unique(labels).tolist())

    # Map each raw cluster ID to its set of languages
    raw_langs = {
        raw: set(languages[labels == raw].tolist())
        for raw in raw_clusters
    }

    target_ids = list(target_representatives.keys())
    raw_to_target = {}
    used_raw = set()

    # First pass: greedily assign raw clusters to targets with best language overlap
    for target_id in target_ids:
        reps = set(target_representatives[target_id])
        candidates = []
        for raw in raw_clusters:
            if raw in used_raw:
                continue
            overlap = len(raw_langs[raw].intersection(reps))
            candidates.append((overlap, -len(raw_langs[raw]), raw))

        if candidates:
            candidates.sort(reverse=True)
            best_overlap, _, best_raw = candidates[0]
            if best_overlap > 0:
                raw_to_target[best_raw] = target_id
                used_raw.add(best_raw)

    # Second pass: deterministically assign any remaining clusters
    remaining_raw = [r for r in raw_clusters if r not in used_raw]
    remaining_targets = [t for t in target_ids if t not in raw_to_target.values()]
    for raw, target in zip(sorted(remaining_raw), remaining_targets):
        raw_to_target[raw] = target

    remapped = np.array([raw_to_target[r] for r in labels])
    return remapped, raw_to_target

# Define representative language sets for 2022 clusters based on original paper semantics
rep_2022 = {
    1: {'Indonesian', 'Korean', 'Portuguese', 'Urdu'},  # Shock & Decay 1
    3: {'Romanian', 'German', 'Dutch', 'Swedish', 'Danish', 'Greek', 'Polish', 'Italian', 'French', 'Finnish', 'Turkish'},  # Shock & Decay 2
    0: {'English', 'Norwegian', 'Catalan', 'Arabic', 'Hungarian'},  # Shock & Sustain 1
    4: {'Persian', 'Spanish'},  # Shock & Sustain 2
    5: {'Czech', 'Estonian', 'Serbian', 'Vietnamese'},  # Noisy Shock & Sustain
    2: {'Ukrainian', 'Russian'}  # Noisy Sustain
}

# Apply stable remapping to 2022 labels; 2014 labels used as-is (already deterministic)
labels_2022, remap_2022 = remap_clusters_by_representatives(
    labels_2022_raw,
    pivot_2022.index,
    rep_2022
)

print('2022 cluster remap (raw -> target):', remap_2022)

def plot_fig3_clusters(df_pivot, labels, cluster_names, desired_order, center_date, tick_freq, save_dir, save_name):
    """Plot trend clusters as a 3×3 subplot grid.
    
    Each subplot shows the normalized log-frequency time-series for all languages
    within a single cluster. Subplots are arranged in the order specified by desired_order.
    
    Args:
        df_pivot: Pivot table (languages × dates) with normalized frequencies
        labels: Cluster assignment for each language (row)
        cluster_names: Dict mapping cluster ID to display name
        desired_order: List of cluster IDs specifying subplot order (top-left to bottom-right)
        center_date: Reference date for x-axis formatting
        tick_freq: Frequency of x-axis ticks (e.g., 'W-TUE' for weekly)
        save_dir: Base directory for output (will create pdf/png/svg subdirs)
        save_name: Base filename for output files
    
    Returns:
        Matplotlib figure object
    """
    df_pivot_log = np.log10(df_pivot + 1)

    # Attach cluster labels by row order
    df_pivot_log['cluster'] = labels

    y_min = df_pivot_log.drop(columns='cluster').min().min()
    y_max = df_pivot_log.drop(columns='cluster').max().max()

    fig, axes = plt.subplots(3, 3, figsize=(13, 13))
    axes = axes.flatten()

    # Set up x-axis ticks: ±4 weeks from center_date
    x_ticks = pd.date_range(
        start=center_date - pd.Timedelta(weeks=4),
        end=center_date + pd.Timedelta(weeks=4),
        freq=tick_freq
    )

    i = -1
    for i, cluster in enumerate(desired_order):
        if cluster in df_pivot_log['cluster'].unique():
            cluster_data = df_pivot_log[df_pivot_log['cluster'] == cluster].drop(columns='cluster')
            cluster_data.columns = pd.to_datetime(cluster_data.columns, errors='coerce')

            ax = axes[i]
            for language in cluster_data.index:
                ax.plot(cluster_data.columns, cluster_data.loc[language], label=language)

            ax.set_title(cluster_names[cluster], fontsize=16)
            ax.legend(loc='upper left', fontsize=10, ncol=2, frameon=False)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels([dt.strftime('%d-%m') for dt in x_ticks], rotation=45, ha='right', fontsize=10)
            ax.tick_params(axis='x', labelsize=14)
            ax.tick_params(axis='y', labelsize=14)
            ax.set_ylim(y_min, y_max)

            # Hide y-axis labels for non-leftmost subplots (to reduce clutter)
            if i % 3 != 0:
                ax.yaxis.set_visible(False)

            ax.xaxis.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)
            ax.yaxis.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

    # Hide empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()

    # Save in multiple formats
    for fmt in ['pdf', 'png', 'svg']:
        out_dir = save_dir / fmt
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / f'{save_name}.{fmt}', format=fmt, dpi=400, bbox_inches='tight')

    return fig

# Define cluster names and display order for 2014
cluster_names_2014 = {
    0: 'Medium Double Shock',
    1: 'Double Shock',
    2: 'Second Peak Shock',
    3: 'Noisy Shock',
    4: 'Weak Noisy Shock'
}
desired_order_2014 = [1, 0, 3, 4, 2]

# Define cluster names and display order for 2022
cluster_names_2022 = {
    0: 'Shock & Sustain 1',
    1: 'Shock & Decay 1',
    2: 'Noisy Sustain',
    3: 'Shock & Decay 2',
    4: 'Shock & Sustain 2',
    5: 'Noisy Shock & Sustain'
}
desired_order_2022 = [1, 4, 0, 2, 5, 3]

# Generate and save Fig.3 for both years
fig_2014 = plot_fig3_clusters(
    pivot_2014_plot, labels_2014, cluster_names_2014, desired_order_2014,
    center_date=CENTRAL_2014_CLUSTER, tick_freq='W-TUE',
    save_dir=FIG3_OUT_2014, save_name='Clusters_2014'
)

fig_2022 = plot_fig3_clusters(
    pivot_2022_plot, labels_2022, cluster_names_2022, desired_order_2022,
    center_date=CENTRAL_2022, tick_freq='W-THU',
    save_dir=FIG3_OUT_2022, save_name='Clusters_2022_with-legend'
)

plt.show()

In [ ]:
# Generate Figure 4: Trend comparison of 2014 vs 2022 frequency curves

def plot_frequency_comparison(df_2014, df_2022):
    """Plot 2014 vs 2022 frequency trends for all 28 languages on log scale.
    
    Creates a 7×4 subplot grid (one for each language). Each subplot overlays:
    - Blue trace: 2014 raw frequencies (shifted temporally to align with 2022)
    - Red trace: 2022 raw frequencies
    - Black dotted line: Feb 24, 2022 (invasion date)
    
    X-axis spans ±4 weeks around the event; y-axis is log-scale to show magnitude changes.
    Gaps appear where frequencies are missing or ≤0 (no interpolation artifacts).
    
    Args:
        df_2014: 2014 data (with date shifted +8y, -3d to overlay with 2022)
        df_2022: 2022 data (centered on Feb 24, 2022)
    
    Returns:
        Plotly figure object
    """
    unique_languages = [lang for lang in FIG4_LANGUAGE_ORDER if lang in set(df_2022['language'])]

    num_languages = len(unique_languages)
    num_rows = (num_languages // 4) + 1
    num_cols = min(num_languages, 4)

    # Set up x-axis ticks and labels relative to Feb 24, 2022
    feb_24_2022 = pd.to_datetime('2022-02-24')
    date_intervals = [(feb_24_2022 - pd.Timedelta(weeks=i)).strftime('%Y-%m-%d') for i in range(4, 0, -1)]
    date_intervals += ['2022-02-24']
    date_intervals += [(feb_24_2022 + pd.Timedelta(weeks=i)).strftime('%Y-%m-%d') for i in range(1, 5)]
    date_labels = ['4 weeks before', '3 weeks before', '2 weeks before', '1 week before', 'Feb 24',
                   '1 week after', '2 weeks after', '3 weeks after', '4 weeks after']

    # Create subplot grid with shared axes
    fig = make_subplots(
        rows=num_rows, cols=num_cols, subplot_titles=unique_languages,
        shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.02, horizontal_spacing=0.02
    )

    # Plot each language
    for i, language in enumerate(unique_languages):
        df_lang_2022 = (
            df_2022[df_2022['language'] == language]
            .sort_values('date')
            .copy()
        )
        df_lang_2014 = (
            df_2014[df_2014['language'] == language]
            .sort_values('date')
            .copy()
        )

        # Mask non-positive values as NaN to avoid interpolation to plot bottom on log scale
        df_lang_2022.loc[df_lang_2022['freq'] <= 0, 'freq'] = np.nan
        df_lang_2014.loc[df_lang_2014['freq'] <= 0, 'freq'] = np.nan

        row = (i // num_cols) + 1
        col = (i % num_cols) + 1

        # Add 2022 trace (red)
        fig.add_trace(
            go.Scatter(
                x=df_lang_2022['date'], y=df_lang_2022['freq'], mode='lines',
                line=dict(color='#bd2f36', width=4), showlegend=False, connectgaps=False
            ),
            row=row, col=col
        )

        # Add 2014 trace (blue, shifted to 2022 dates)
        fig.add_trace(
            go.Scatter(
                x=df_lang_2014['date'], y=df_lang_2014['freq'], mode='lines',
                line=dict(color='#4d99c6', width=4), showlegend=False, connectgaps=False
            ),
            row=row, col=col
        )

        # Add vertical line at invasion date (Feb 24, 2022)
        fig.add_vline(x='2022-02-24', line=dict(color='black', width=2, dash='dot'), row=row, col=col)

    # Configure layout and axes
    fig.update_layout(
        title='',
        showlegend=False,
        margin=dict(l=10, r=10, t=40, b=10),
        height=2718,
        width=2127,
        font=dict(size=30)
    )

    fig.update_annotations(font=dict(size=32))

    # Y-axis: log scale with fixed range
    fig.update_yaxes(
        scaleanchor='x', scaleratio=1, type='log', showline=True,
        linewidth=1, linecolor='black', mirror=True,
        tickvals=[1e-6, 1e-5, 1e-4, 1e-3], range=[-7, -2]
    )

    # X-axis: custom labels and formatting
    fig.update_xaxes(
        constrain='domain', showline=True, linewidth=1, linecolor='black', mirror=True,
        tickvals=date_intervals, ticktext=date_labels
    )

    fig.update_xaxes(gridcolor='#cccccc', gridwidth=1.1)
    fig.update_yaxes(gridcolor='#cccccc', gridwidth=1.1)

    return fig

# Generate Fig.4 and save in multiple formats
fig4 = plot_frequency_comparison(DF2014_fig4, DF2022_fig4)
fig4.show()

for fmt in ['pdf', 'jpeg', 'svg']:
    out_dir = FIG4_OUT / fmt
    out_dir.mkdir(parents=True, exist_ok=True)
    fig4.write_image(out_dir / f'2014vs2022_trends_4w_notnormalized_daily.{fmt}', format=fmt, engine='kaleido')

print('Saved Fig.4 files.')